# 矩阵乘法介绍

矩阵乘法是深度学习与科学计算中最核心的运算之一。在昇腾AI处理器上，Ascend C 提供了一套高效、灵活的矩阵编程接口。本文将系统介绍矩阵乘法的基础知识，包括计算原理、数据格式、编程范式等内容，为后续使用Ascend C编程实现高性能MatMul算子的完整方法提供坚实的理论依据。

本节学习大纲如下：
- 矩阵乘法基础
- 矩阵乘相关分形格式
- 矩阵编程范式

---

## 1. 矩阵乘法基础

矩阵乘法是深度学习与高性能计算中的核心运算之一, 矩阵乘（MatMul）的基本计算公式为：

C = A × B + Bias

其中：

- A（左矩阵）：形状为 [M, K]

- B（右矩阵）：形状为 [K, N]

- C（结果矩阵）：形状为 [M, N]

- Bias（偏置向量）：形状为 [N]，在推理或全连接层中常用

在 Ascend C 中，该公式通过 Matmul 对象实现，支持半精度（half）和单精度（float）数据类型。

<img src="./images/matrix_multiplication.png" alt="矩阵乘法"  width="700px" >

---

## 2. 矩阵乘相关分形格式

数据排布格式（Data Layout Format）是深度学习中对多维Tensor在内存中存储方式的描述。常见的格式包括 NHWC 和 NCHW，它们为张量的每个维度赋予了特定的语义（如批大小、通道、高度、宽度）。除了这些通用格式外，为了充分发挥 AI 计算硬件（如 Ascend AI Core 中的 Cube 计算单元）的并行计算能力，Ascend C还引入了一系列特殊的分形格式，如 FRACTAL_NZ（简称 NZ）、FRACTAL_ZZ、NC1HWC0 等。这类格式通过重塑数据在内存中的排列方式，显著提升了矩阵乘、卷积等计算密集型运算的效率。

**为什么需要分形格式？**

AI Core 中的 Cube 单元是专为矩阵运算优化的硬件模块，其计算模式并非逐元素操作，而是每次以小数据块(16×16×16)为单位进行并行计算（以half数据类型为例）。为了在一个时钟周期内高效地为计算单元提供数据，内存中的数据必须满足以下条件：
 
- **连续访问**：计算所需的数据块应尽量连续存储，以最大化内存带宽利用率。

- **数据复用**：合理安排数据布局，使已加载的数据能在多次计算中被重复使用，减少数据搬运开销。

传统的 ND（行优先/列优先）格式虽然适合 CPU 的缓存访问模式，但在面对 Cube 单元的块计算时，数据往往呈现跳跃式分布，导致访存延迟增加、效率降低。为此，分形格式通过数据重组，实现了计算数据在物理内存中的“对齐”。

使用Mmad基础API进行矩阵乘计算时，对矩阵输入输出的数据排布格式有一定的要求，如下图所示，要求A矩阵（位于L0A Buffer）为FRACTAL_ZZ，B矩阵（位于L0B Buffer）为FRACTAL_ZN，C矩阵（位于L0C Buffer）为FRACTAL_NZ。这些格式将矩阵划分成了一些分形（Fractal Matrix），适配Cube计算单元每次读取(16, 16)× (16, 16) 的数据进行计算的硬件特点（以half数据类型为例），从而提高矩阵计算的效率。分形的大小和数据类型有关，也和所在的存储位置有关。

<img src="./images/nz_format.png" alt="NZ格式"  width="700px" >

### 2.1 FRACTAL_NZ/NZ

FRACTAL_NZ格式，简称NZ格式，是对一个Tensor最低两维（一个Tensor的所有维度，右侧为低维，左侧为高维）进行填充（pad）、拆分（reshape）和转置（transpose）操作后得到的格式。具体的转换过程如下：

(M，N)大小的矩阵被分为M1 * N1个分形，按照column major（列优先）排布，形状如N字形；每个分形内部有M0 * N0个元素，按照row major（行优先）排布，形状如Z字形，所以这种数据格式称为NZ格式。其中，(M0, N0)表示一个分形的大小。

<img src="./images/FRACTAL_NZ.png" alt="FRACTAL_NZ"  width="200px" >

通过公式表达为：

```
(…, B, M, N)->pad->(…, B, M1 * M0, N1 * N0)->reshape->(…, B, M1, M0, N1, N0)->transpose->(…, B, N1, M1, M0, N0)
```

**存储示例**

假设分形大小为 2×2，原始 16 个元素按行优先（ND）存储为：

`0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15`

转换为 NZ 格式后，数据被重组为：

`0, 1, 4, 5, 8, 9, 12, 13, 2, 3, 6, 7, 10, 11, 14, 15`

这种排列使得在计算相邻分形块时，所需数据在物理内存中也保持相邻，极大提高了 Cube 单元的数据吞吐效率。

<img src="./images/ND2NZ.png" alt="ND2NZ"  width="700px" >


### 2.2 FRACTAL_ZZ/ZZ

FRACTAL_ZZ格式，简称ZZ格式，是对一个Tensor最低两维（一个Tensor的所有维度，右侧为低维，左侧为高维）进行填充（pad）、拆分（reshape）和转置（transpose）操作后得到的格式。具体转换过程如下：

(M, K)大小的矩阵被分为M1 * K1个分形，按照row major排布，形状如Z字形；每个分形内部有M0 * K0个元素，按照row major排布，形状如Z字形，所以这种数据格式称为ZZ格式。其中，(M0, K0)表示一个分形的大小，分形Shape为16 x (32B / sizeof(Datatype))，大小为512字节。

<img src="./images/FRACTAL_ZZ.png" alt="FRACTAL_ZZ"  width="200px" >

通过公式表达转换过程如下：
```
(…, B, M, K)->pad->(…, B, M1 * M0, K1 * K0)->reshape->(…, B, M1, M0, K1, K0)->transpose->(…, B, M1, K1, M0, K0)
```

对于不同的数据类型，M0和K0的大小不同：

- 位宽为4的数据类型：M0=16，K0=64。
- 位宽为8的数据类型：M0=16，K0=32。
- 位宽为16的数据类型：M0=16，K0=16。
- 位宽为32的数据类型，M0=16，K0=8。

### 2.3 FRACTAL_ZN/ZN

FRACTAL_ZN格式，简称ZN格式，是对一个Tensor最低两维（一个Tensor的所有维度，右侧为低维，左侧为高维）进行填充（pad）、拆分（reshape）和转置（transpose）操作后得到的格式。具体转换过程如下：

(K, N)大小的矩阵被分为K1 * N1个分形，按照row major排布，形状如Z字形；每个分形内部有K0 * N0个元素，按照column major排布，形状如N字形，所以这种数据格式称为ZN格式。其中，(K0, N0)表示一个分形的大小，分形shape为 (32B / sizeof(Datatype)) x 16，大小为512字节。


<img src="./images/FRACTAL_ZN.png" alt="FRACTAL_ZN"  width="200px" >

通过公式表达转换过程如下：

```
(…, B, K, N)->pad->(…, B, K1 * K0, N1 * N0)->reshape->(…, B, K1, K0, N1, N0)->transpose->(…, B, K1, N1, N0, K0)
```

对于不同的数据类型，K0和N0的大小不同：

- 位宽为4的数据类型：K0=64，N0=16；
- 位宽为8的数据类型：K0=32，N0=16；
- 位宽为16的数据类型：K0=16，N0=16；
- 位宽为32的数据类型：K0=8，N0=16。

分形格式（如 NZ）通过硬件友好的数据重排，解决了传统 ND 格式在矩阵块计算中访存不连续、数据复用率低的问题。它不仅适应了 AI Core 的并行计算特性，也为实现高性能算子（如矩阵乘、卷积）提供了关键的内存布局基础。在 Ascend C 编程中，正确使用 FRACTAL_ZZ、FRACTAL_ZN、FRACTAL_NZ 等对应格式，是发挥硬件算力的重要一环。

---



## 3. 矩阵乘法数据流

在了解矩阵乘法数据流之前，需要先回顾一下几个重要的存储逻辑位置的概念：

- 搬入数据的存放位置：A1，用于存放整块A矩阵，可类比CPU多级缓存中的二级缓存；
- 搬入数据的存放位置：B1，用于存放整块B矩阵，可类比CPU多级缓存中的二级缓存；
- 搬入数据的存放位置：C1，用于存放整块的矩阵乘偏置Bias矩阵，可类比CPU多级缓存中的二级缓存；
- 搬入数据的存放位置：A2，用于存放切分后的小块A矩阵，可类比CPU多级缓存中的一级缓存；
- 搬入数据的存放位置：B2，用于存放切分后的小块B矩阵，可类比CPU多级缓存中的一级缓存；
- 搬入数据的存放位置：C2，用于存放切分后的小块矩阵乘偏置Bias矩阵，可类比CPU多级缓存中的一级缓存；
- 结果数据的存放位置：CO1，用于存放小块结果C矩阵，可理解为Cube Out；
- 结果数据的存放位置：CO2，用于存放整块结果C矩阵，可理解为Cube Out；
- 搬入数据的存放位置：VECCALC，一般在计算需要临时变量时使用此位置。

**矩阵乘法数据流**指矩阵乘的输入输出在各存储位置间的流向。逻辑位置的数据流向如下图所示（为了简化描述，没有列出bias）：

- A矩阵从输入位置到A2的数据流如下（输入位置可以是GM或者VECOUT）：GM->A2，GM->A1->A2；VECOUT->A1->A2。

    由于A1比A2的空间更大，数据从GM或VECOUT可以先搬入A1进行缓存，待该数据执行Cube计算前，数据直接从A1搬入A2，这样在搬运大量数据时可以减少计算前的等待时间，提升性能，只有在搬入数据较少的场景才可能使用GM->A2的数据流。

- B矩阵从输入位置到B2的数据流如下（输入位置可以是GM或者VECOUT）：GM->B2，GM->B1->B2；VECOUT->B1->B2。

    由于B1比B2的空间更大，数据从GM或VECOUT可以先搬入B1进行缓存，待该数据执行Cube计算前，数据直接从B1搬入B2，这样在搬运大量数据时可以减少计算前的等待时间，提升性能，只有在搬入数据较少的场景才可能使用GM->B2的数据流。

- 完成A2*B2=CO1计算。
- CO1数据汇聚到CO2：CO1->CO2。
- 从CO2到输出位置（输出位置可以是GM或者VECIN）：CO2->GM/CO2->VECIN。

<img src="./images/matrix_programming_paradigm.png" alt="矩阵编程范式"  width="700px" >

Cube计算流程同样也可以理解为CopyIn、Compute、CopyOut这几个阶段，因为流程相对复杂，Matmul高阶API提供对此的高阶封装，简化了编程范式。

<img src="./images/simplified_matrix_programming_paradigm.png" alt="矩阵简化编程范式" width="700px">

如上图所示：

- CopyIn阶段对应SetTensorA、SetTensorB、SetBias接口；
- Compute阶段对应Iterate接口；
- CopyOut阶段对应GetTensorC接口。

在实际开发中，使用Matmul高阶API实现矩阵乘算子的核函数遵循以下五个标准步骤：

1. 创建Matmul对象

    实例化Matmul模板类，并通过MatmulType指定输入输出矩阵的数据类型、存储位置与数据格式。

2. 初始化

    通过REGIST_MATMUL_OBJ宏将Matmul对象与TPipe及Tiling参数绑定，完成内部资源准备。

3. 设置左矩阵A、右矩阵B、Bias

    调用SetTensorA、SetTensorB、SetBias接口，将当前AI Core负责的左矩阵A、右矩阵B及偏置Bias的全局内存视图传递给Matmul对象。

4. 执行矩阵乘操作
    
    调用Iterate（灵活控制迭代）或IterateAll（全量计算）接口启动矩阵乘计算，并通过GetTensorC获取结果。

5. 结束矩阵乘操作

    调用End接口释放Matmul计算资源。

典型代码示例如下：

```
// 创建Matmul对象 创建对象时需要传入A、B、C、Bias的参数类型信息， 类型信息通过MatmulType来定义，包括：内存逻辑位置、数据格式、数据类型。
typedef MatmulType<TPosition::GM, CubeFormat::ND, half> aType; 
typedef MatmulType<TPosition::GM, CubeFormat::ND, half> bType; 
typedef MatmulType<TPosition::GM, CubeFormat::ND, float> cType; 
typedef MatmulType<TPosition::GM, CubeFormat::ND, float> biasType; 
Matmul<aType, bType, cType, biasType> mm; 

REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), mm, &tiling); // 初始化
// CopyIn阶段：完成从GM到LocalMemory的搬运
mm.SetTensorA(gm_a);    // 设置左矩阵A
mm.SetTensorB(gm_b);    // 设置右矩阵B
mm.SetBias(gm_bias);    // 设置Bias
// Compute阶段：完成矩阵乘计算
while (mm.Iterate()) { 
    // CopyOut阶段：完成从LocalMemory到GM的搬运
    mm.GetTensorC(gm_c); 
}
// 结束矩阵乘操作
mm.End();
```

### 3.1 Matmul高阶API详解

Matmul高阶API封装了矩阵乘的核心流程，开发者只需调用几个关键接口即可完成数据搬移、计算和结果输出。常用接口及其功能如下：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <td align="center"><strong>API</strong></td>
  <td align="center"><strong>功能描述</strong></td>
</tr>
<tr>
  <td align="left">REGIST_MATMUL_OBJ</td>
  <td align="left">初始化Matmul对象。</td>
</tr>
<tr>
  <td align="left">SetTensorA</td>
  <td align="left">设置矩阵乘的左矩阵A。</td>
</tr>
<tr>
  <td align="left">SetTensorB</td>
  <td align="left">设置矩阵乘的右矩阵B。</td>
</tr>
<tr>
  <td align="left">SetBias</td>
  <td align="left">设置矩阵乘的Bias。</td>
</tr>
<tr>
  <td align="left">Iterate</td>
  <td align="left">每调用一次Iterate，会计算出一片baseM * baseN的C矩阵。</td>
</tr>
<tr>
  <td align="left">IterateAll</td>
  <td align="left">调用一次IterateAll，会计算出singleCoreM * singleCoreN大小的C矩阵。</td>
</tr>
<tr>
  <td align="left">GetTensorC</td>
  <td align="left">配合Iterate使用，一次Iterate后，获取一块或者两块C矩阵片，可以直接输出到GM tensor中，也可以输出到VECIN tensor中。</td>
</tr>
<tr>
  <td align="left">End</td>
  <td align="left">一个matmul计算结束时需要调用一次End函数。</td>
</tr>
<tr>
  <td align="left">SetTail</td>
  <td align="left">在不改变Tiling的情况下，重新设置本次计算的singleCoreM/singleCoreN/singleCoreK。</td>
</tr>
</table>

这些API的设计目标是将复杂的五级流水（CopyIn→Split→Compute→Aggregate→CopyOut）简化为CopyIn、Compute、CopyOut三个主要阶段，开发者无需手动管理分形块的切分与排布。

- CopyIn阶段：SetTensorA、SetTensorB、SetBias负责将数据从Global Memory搬运到Local Memory，并自动完成格式转换（如ND转NZ）。

- Compute阶段：Iterate或IterateAll触发Cube单元执行分形块矩阵乘，同时累加偏置（若已设置）。每次迭代计算一个或多个分形块，具体数量由Tiling参数决定。

- CopyOut阶段：GetTensorC获取计算结果在L0C上的地址，开发者需将其搬回Global Memory，并可根据需要转换回ND格式。

通过Tiling参数（在初始化时通过REGIST_MATMUL_OBJ绑定），开发者可以精细控制分块大小、迭代次数以及每个核负责的计算范围，从而适配不同规模的矩阵和硬件资源。

### 3.2 数据分块（Tiling）

- 多核切分

    为了实现多核并行，需要将矩阵数据进行切分，分配到不同的核上进行处理。切分策略如下图所示：

   - 对于A矩阵，沿着M轴进行切分，切分成多份的singleCoreM，单核上处理SingleCoreM * K大小的数据。
    - 对于B矩阵，沿着N轴进行切分，切分成多份的singleCoreN，单核上处理K * SingleCoreN大小的数据。
    - 对于C矩阵，SingleCoreM * K大小的A矩阵和K * SingleCoreN大小的B矩阵相乘得到SingleCoreM * SingleCoreN大小的C矩阵，即为单核上输出的C矩阵大小。
    
    比如，下图中共有8个核参与计算，将A矩阵沿着M轴划分为4块，将B矩阵沿着N轴切分为两块，单核上仅处理某一分块（比如图中绿色部分为core3上参与计算的数据）：SingleCoreM * K大小的A矩阵分块和SingleCoreN* K大小的B矩阵分块相乘得到SingleCoreM * SingleCoreN大小的C矩阵分块。

    <img src="./images/Multi-core_splitting.png" alt="多核切分示意图"  width="700px" >   
 
    另外，单核上处理的K轴长度为SingleCoreK，对于K轴较大的场景，可以沿着K轴进行切分，切分成多份的singleCoreK。

- 核内切分

    大多数情况下，Local Memory的存储，无法完整的容纳算子的输入与输出，需要每次搬运一部分输入进行计算然后搬出，再搬运下一部分输入进行计算，直到得到完整的最终结果，也就是需要做核内的输入切分。切分的策略如下所示：

    - 对于A矩阵，沿M轴进行切分，将singleCoreM切分成多份的baseM，切分成的份数对应图示的mIter；沿K轴进行切分，切分成多份的baseK。
    - 对于B矩阵，沿N轴进行切分，将singleCoreN切分成多份的baseN，切分成的份数对应图示的nIter；沿K轴进行切分，切分成多份的baseK。
    - 对于C矩阵，A矩阵中baseM * baseK大小的分块和B矩阵中baseK * baseN大小的分块相乘并累加，得到C矩阵中对应位置baseM * baseN大小的分块。比如，图中结果矩阵中的绿色矩阵块5是通过如下的累加过程得到的：a * a + b * b + c * c + d * d + e * e + f * f。
    
    <img src="./images/single_core_splitting.png" alt="单核切分示意图"  width="700px" >
    
    除了基本块形状baseM, baseN, baseK外，还有一些常用的tiling参数，其含义如下：

    - iterateOrder：一次Iterate迭代计算出[baseM, baseN]大小的C矩阵分片。Iterate完成后，Matmul会自动偏移下一次Iterate输出的C矩阵位置，iterateOrder表示自动偏移的顺序。
        - 0代表先往M轴方向偏移再往N轴方向偏移；
        - 1代表先往N轴方向偏移再往M轴方向偏移。
        - 在上图的示例中，iterateOrder取值为0。

    - depthA1，depthB1：A1、B1上存储的矩阵片全载A2、B2的份数，A2、B2存储大小分别是baseM * baseK、baseN * baseK，即depthA1是A1矩阵切片含有baseM * baseK块的个数，depthB1是B1矩阵切片含有baseN * baseK块的个数。
    - stepM，stepN：stepM为左矩阵在A1中缓存的buffer M方向上baseM的倍数，stepN为右矩阵在B1中缓存的buffer N方向上baseN的倍数。
    - stepKa，stepKb：stepKa为左矩阵在A1中缓存的buffer K方向上baseK的倍数，stepKb为右矩阵在B1中缓存的buffer K方向上baseK的倍数。

    

---

## 课后练习

请根据本节课程学习内容完成以下题目进行自测。

1. 关于 Matmul 高阶 API 中的 Iterate 与 IterateAll 接口，下列说法正确的是？

    A. Iterate 每次计算一整块 C 矩阵，大小等于 singleCoreM × singleCoreN
    
    B. IterateAll 每次仅计算一个 baseM × baseN 的 C 矩阵分片
    
    C. Iterate 需要配合 GetTensorC 在循环内多次调用，逐步获取结果分片
    
    D. IterateAll 调用后不需要再调用 End 接口释放资源

2. 将传统行优先（ND）格式的张量转换为 FRACTAL_NZ 分形格式时，正确的数据重组步骤顺序是？

    A. 填充 -> 拆分维度 -> 转置重组

    B. 拆分维度 -> 转置重组 -> 填充

    C. 转置重组 -> 填充 -> 拆分维度

    D. 拆分维度 -> 填充 -> 转置重组

3. 在 Ascend C 的矩阵编程范式中，以下哪个“逻辑位置”与物理存储 “L0A Buffer” 相对应，并用于存放小块左矩阵？

    A. A1

    B. A2

    C. B2

    D. CO1

4. 使用 Ascend C 的 Matmul 高阶 API 简化编程时，以下哪个接口组合正确地对应了 “CopyIn -> Compute -> CopyOut” 的完整流程？

    A. SetTensorA -> Iterate -> SetBias

    B. SetTensorB -> GetTensorC -> Iterate

    C. Iterate -> GetTensorC -> SetTensorA

    D. SetTensorA/SetTensorB/SetBias -> Iterate -> GetTensorC

5. 分形格式（如 FRACTAL_NZ）被引入 Ascend C 的主要目的是什么？

    A. 为了统一不同深度学习框架（如TensorFlow, PyTorch）的数据格式标准

    B. 主要目的是减少存储空间占用，对计算性能没有直接影响

    C. 解决传统 ND 格式在 AI Core Cube 单元块计算时访存不连续、数据复用率低的问题

    D. 为了让程序员更直观地理解和编写矩阵运算代码

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/04.02_answer.txt